In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import pandas
import numpy
import seaborn as sns
from tensorflow import keras
from keras.layers import Input, Dense,InputLayer,BatchNormalization
from keras.models import Sequential
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint,ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder


from sklearn.preprocessing import StandardScaler




In [ ]:
train_data=pandas.read_csv('/kaggle/input/playground-series-s3e23/train.csv')
test_data=pandas.read_csv('/kaggle/input/playground-series-s3e23/test.csv')

In [ ]:
train_data.shape

In [ ]:
train_data.head()

In [ ]:
train_data.describe()

In [ ]:
(train_data.isna().sum()/len(train_data.index))*100

In [ ]:
for col in train_data.columns:
    print("for {col} number of {unique}".format(col=col,unique=train_data[col].nunique()))

In [ ]:
train_data.drop(columns=['id'],inplace=True)

In [ ]:
arr=list(train_data.columns)
arr.remove('defects')

In [ ]:
train_data=train_data.drop_duplicates()

In [ ]:
def remove_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

In [ ]:
def plot_features(col):
    plt.figure(figsize=(10, 6))
    
    sns.histplot(train_data[col],kde=True)
    plt.show()
   

In [ ]:
from scipy.stats import boxcox


for x in arr:
    #Performing log transformation as the data is skewed to right
    train_data[x]=numpy.log1p(train_data[x])
    plot_features(x)

In [ ]:
label_encoder = LabelEncoder()
train_data['defects'] = label_encoder.fit_transform(train_data['defects'])


In [ ]:
model=Sequential()
model.add(InputLayer(input_shape=(21)))
model.add(BatchNormalization())
model.add(Dense(512,activation='relu'))
model.add(Dense(512,activation='relu'))
model.add(Dense(1,activation='sigmoid'))


In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
Y=train_data.pop('defects')


In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(train_data,Y,test_size=0.2,stratify=Y)

In [ ]:
print(X_train.shape)
print(Y_train.shape)
print(X_test.shape)
print(Y_test.shape)

In [ ]:
checkpoint = ModelCheckpoint(filepath='/kaggle/working/best_mode.h5', 
                             monitor="val_accuracy",
                             verbose=1, 
                             save_best_only=True,
                             mode="max")

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', 
                              factor=0.1,  
                              patience=5,  
                              min_lr=0.00001)

In [ ]:
history = model.fit(X_train, Y_train, epochs=100, batch_size=32, validation_data=(X_test, Y_test),callbacks=[checkpoint,reduce_lr])


In [ ]:
model=load_model('/kaggle/working/best_mode.h5')

In [ ]:
train_acc = history.history['accuracy'][-1]
train_acc


In [ ]:
val_acc = history.history['val_accuracy'][-1]
val_acc


In [ ]:
test_data.head()

In [ ]:
test_data.describe()

In [ ]:
test_data.isna().sum()/len(test_data.index)

In [ ]:
id=test_data.pop('id')

In [ ]:
train_data=test_data.drop_duplicates()

In [ ]:
col=list(test_data.columns)
for x in col:
    test_data[x]=numpy.log1p(test_data[x])

In [ ]:
pred=model.predict(test_data)

In [ ]:
numpy.squeeze(pred)

In [ ]:
fin_df=pandas.DataFrame(data=zip(id,numpy.squeeze(pred)),columns=['id','defects'])

In [ ]:
fin_df

In [ ]:
fin_df.to_csv('/kaggle/working/submission.csv',index=False)